# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Random Forest — Health Score

The Random Forest model reports that Average Position (43%), Impressions (32%), and Scroll Depth (15%) are the most important features when predicting the Health Score.

The target is a composite score constructed from Impressions (30 points), Average Position (30 points), CTR (20 points), and Scroll Depth (20 points). Therefore, the model uses features that directly contribute to its target. This is target leakage by construction, even with a holdout split. The feature importance results describe how the model reconstructs the score, not independent factors that cause better health.

### Finding 2: Logistic Regression — Growth Classification

The Logistic Regression model reports 71% holdout accuracy when separating growing from declining pages. Content Age is reported as the strongest negative signal, while Days Visible and recent impressions are among the strongest positive signals.

The paper mentions a 30-day trend comparison, but does not clearly specify the exact label-generation rule or the time alignment between the features and the label window. It also reports an 80/20 holdout split without specifying whether it is grouped by client or separated by time. Therefore, the result is exploratory and does not establish reliable performance on unseen clients or future data.

### Methodology questions

The paper should clarify the exact growth/decline label definition, ensure that all model features are available before the label window, and specify whether validation uses grouped or time-based splits. A base-rate comparison would also help establish whether 71% accuracy provides meaningful improvement.

In [5]:
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Load the dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Recreate the Week 5 target
df = df[df["trend_direction"] != "new"].copy()
y = (df["trend_direction"] == "down").astype(int)

# Recreate the Week 5 feature set
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
    "provider_used",
    "model_used",
    "age_tier",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier"
]

feature_columns = numeric_features + categorical_features
X = df[feature_columns]
groups = df["client_id"]

# Build the same preprocessing and model pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Grouped split by client
splitter_w6 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx_w6, test_idx_w6 = next(
    splitter_w6.split(X, y, groups=groups)
)

X_train_w6 = X.iloc[train_idx_w6]
X_test_w6 = X.iloc[test_idx_w6]
y_train_w6 = y.iloc[train_idx_w6]
y_test_w6 = y.iloc[test_idx_w6]

model.fit(X_train_w6, y_train_w6)

# Evaluate on eligible test pages
evaluation_w6 = pd.DataFrame(index=X_test_w6.index)
evaluation_w6["actual"] = y_test_w6
evaluation_w6["model_score"] = model.predict_proba(X_test_w6)[:, 1]
evaluation_w6["baseline_score"] = df.loc[
    X_test_w6.index, "impressions_90d"
]

evaluation_w6 = evaluation_w6[
    evaluation_w6["baseline_score"] >= 3000
].copy()

model_top_w6 = evaluation_w6.nlargest(50, "model_score")
baseline_top_w6 = evaluation_w6.nlargest(50, "baseline_score")

# Report results
train_clients = set(groups.iloc[train_idx_w6])
test_clients = set(groups.iloc[test_idx_w6])

print("Train rows:", len(X_train_w6))
print("Test rows:", len(X_test_w6))
print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))
print("Eligible test pages:", len(evaluation_w6))
print("Model Precision@50:", round(model_top_w6["actual"].mean(), 3))
print("Baseline Precision@50:", round(baseline_top_w6["actual"].mean(), 3))
print("Test base rate:", round(evaluation_w6["actual"].mean(), 3))

Train rows: 22172
Test rows: 5592
Train clients: 24
Test clients: 7
Client overlap: 0
Eligible test pages: 1315
Model Precision@50: 0.6
Baseline Precision@50: 0.44
Test base rate: 0.434


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Audit the final Week 5 feature set for leakage risks

label_columns = ["trend_direction", "trend_pct"]
id_columns = ["content_id", "client_id"]

audit_checks = pd.DataFrame([
    {
        "check": "Label-derived columns",
        "columns": ", ".join(label_columns),
        "result": "PASS" if not set(label_columns) & set(feature_columns) else "FAIL"
    },
    {
        "check": "Identifier columns",
        "columns": ", ".join(id_columns),
        "result": "PASS" if not set(id_columns) & set(feature_columns) else "FAIL"
    },
    {
        "check": "Feature count",
        "columns": str(len(feature_columns)),
        "result": "PASS"
    }
])

display(audit_checks)

# Identify features that represent derived versions of other features
derived_features = {
    "age_tier": "content_age_days",
    "freshness_tier": "days_since_last_update",
    "word_count_tier": "word_count",
    "char_count_tier": "char_count"
}

derived_audit = pd.DataFrame([
    {"derived_feature": derived, "source_feature": source}
    for derived, source in derived_features.items()
    if derived in feature_columns and source in feature_columns
])

display(derived_audit)

,check,columns,result
0,Label-derived columns,"trend_direction, trend_pct",PASS
1,Identifier columns,"content_id, client_id",PASS
2,Feature count,16,PASS


,derived_feature,source_feature
0,age_tier,content_age_days
1,freshness_tier,days_since_last_update
2,word_count_tier,word_count
3,char_count_tier,char_count


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

**Original claim:**

The Logistic Regression model predicts declining content with 60% Precision@50, outperforming the impressions-based baseline at 44%.

**Rewritten claim:**

On the held-out set of seven unseen clients, the Logistic Regression model achieved 60% Precision@50 among eligible pages, compared with 44% for the impressions-based baseline. The test-set decline base rate was 43.4%. These results suggest that the model may support prioritizing pages for review within this dataset. However, they do not establish future predictive performance or causal relationships, and temporal separation between features and the observed trend label has not been verified.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.